In [55]:
!pip install torch transformers scikit-learn pandas tqdm telethon pymorphy2 emoji nltk

import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from tqdm.notebook import tqdm
from telethon.sync import TelegramClient
import re
import pymorphy2
import emoji
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk
from collections import Counter

In [56]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

MODEL_NAME = "DeepPavlov/rubert-base-cased"
BATCH_SIZE = 32
EPOCHS = 6
MAX_LEN = 128
CSV_FILE = "dataset_clean.csv"

Device: cuda


In [57]:
df = pd.read_csv(CSV_FILE, encoding='utf-8-sig')
df = df.dropna(subset=['text'])

le = LabelEncoder()
df['label'] = le.fit_transform(df['category'])
num_classes = len(le.classes_)
print(f"Категории: {le.classes_}")
print(f"Количество классов: {num_classes}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenized = tokenizer(
    df['text'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=MAX_LEN,
    return_tensors='pt'
)

input_ids = tokenized['input_ids']
attention_mask = tokenized['attention_mask']
labels = torch.tensor(df['label'].values, dtype=torch.long)

dataset = TensorDataset(input_ids, attention_mask, labels)

val_size = int(0.1 * len(dataset))
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

Категории: ['IT' 'Блоги' 'Игры' 'Новости' 'Образование' 'Политика' 'Путешествия'
 'Финансы']
Количество классов: 8


In [58]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_classes)
model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1

In [61]:
class_counts = df['label'].value_counts().sort_index().values
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
class_weights = class_weights / class_weights.sum() * num_classes
class_weights = class_weights.to(device)

criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

scaler = torch.cuda.amp.GradScaler()

C:\Users\kiru\AppData\Local\Temp\ipykernel_36356\2000545197.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [64]:
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for i, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")):
        optimizer.zero_grad()
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        with torch.cuda.amp.autocast():
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss += loss.item()
        if i % 50 == 0:
            tqdm.write(f"Batch {i}/{len(train_loader)} - Loss: {loss.item():.4f}")

    avg_loss = train_loss / len(train_loader)
    print(f"Epoch {epoch+1} train loss: {avg_loss:.4f}")

    model.eval()
    val_labels = []
    val_preds = []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch[0].to(device)
            attention_mask = batch[1].to(device)
            labels = batch[2].to(device)
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(val_labels, val_preds)
    f1 = f1_score(val_labels, val_preds, average='weighted')
    print(f"Validation Accuracy: {acc:.4f}, F1-score: {f1:.4f}")

Epoch 1/6:   0%|          | 0/1236 [00:00<?, ?it/s]

C:\Users\kiru\AppData\Local\Temp\ipykernel_36356\852288039.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Batch 0/1236 - Loss: 2.0668
Batch 50/1236 - Loss: 2.0441
Batch 100/1236 - Loss: 1.7996
Batch 150/1236 - Loss: 1.5796
Batch 200/1236 - Loss: 1.4084
Batch 250/1236 - Loss: 1.1454
Batch 300/1236 - Loss: 0.8645
Batch 350/1236 - Loss: 0.6749
Batch 400/1236 - Loss: 0.6908
Batch 450/1236 - Loss: 0.6547
Batch 500/1236 - Loss: 0.2934
Batch 550/1236 - Loss: 0.5451
Batch 600/1236 - Loss: 0.6895
Batch 650/1236 - Loss: 0.7818
Batch 700/1236 - Loss: 0.5974
Batch 750/1236 - Loss: 0.4710
Batch 800/1236 - Loss: 0.4822
Batch 850/1236 - Loss: 0.5854
Batch 900/1236 - Loss: 0.6212
Batch 950/1236 - Loss: 0.4934
Batch 1000/1236 - Loss: 0.3197
Batch 1050/1236 - Loss: 0.5498
Batch 1100/1236 - Loss: 0.2773
Batch 1150/1236 - Loss: 0.7465
Batch 1200/1236 - Loss: 0.4742
Epoch 1 train loss: 0.8096
Validation Accuracy: 0.8358, F1-score: 0.8370


Epoch 2/6:   0%|          | 0/1236 [00:00<?, ?it/s]

C:\Users\kiru\AppData\Local\Temp\ipykernel_36356\852288039.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


...

C:\Users\kiru\AppData\Local\Temp\ipykernel_36356\852288039.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Batch 0/1236 - Loss: 0.0341
Batch 50/1236 - Loss: 0.0146
Batch 100/1236 - Loss: 0.0782
Batch 150/1236 - Loss: 0.0303
Batch 200/1236 - Loss: 0.0067
Batch 250/1236 - Loss: 0.0326
Batch 300/1236 - Loss: 0.0124
Batch 350/1236 - Loss: 0.1269
Batch 400/1236 - Loss: 0.0676
Batch 450/1236 - Loss: 0.0060
Batch 500/1236 - Loss: 0.0984
Batch 550/1236 - Loss: 0.1415
Batch 600/1236 - Loss: 0.1686
Batch 650/1236 - Loss: 0.0281
Batch 700/1236 - Loss: 0.0681
Batch 750/1236 - Loss: 0.0409
Batch 800/1236 - Loss: 0.0082
Batch 850/1236 - Loss: 0.2791
Batch 900/1236 - Loss: 0.0139
Batch 950/1236 - Loss: 0.0095
Batch 1000/1236 - Loss: 0.0261
Batch 1050/1236 - Loss: 0.1278
Batch 1100/1236 - Loss: 0.0038
Batch 1150/1236 - Loss: 0.0152
Batch 1200/1236 - Loss: 0.1569
Epoch 5 train loss: 0.0870
Validation Accuracy: 0.8711, F1-score: 0.8711


Epoch 6/6:   0%|          | 0/1236 [00:00<?, ?it/s]

C:\Users\kiru\AppData\Local\Temp\ipykernel_36356\852288039.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Batch 0/1236 - Loss: 0.0094
Batch 50/1236 - Loss: 0.0461
Batch 100/1236 - Loss: 0.0107
Batch 150/1236 - Loss: 0.0098
Batch 200/1236 - Loss: 0.0366
Batch 250/1236 - Loss: 0.1936
Batch 300/1236 - Loss: 0.0025
Batch 350/1236 - Loss: 0.0127
Batch 400/1236 - Loss: 0.0031
Batch 450/1236 - Loss: 0.1089
Batch 500/1236 - Loss: 0.0051
Batch 550/1236 - Loss: 0.0060
Batch 600/1236 - Loss: 0.0362
Batch 650/1236 - Loss: 0.0431
Batch 700/1236 - Loss: 0.0299
Batch 750/1236 - Loss: 0.2228
Batch 800/1236 - Loss: 0.0418
Batch 850/1236 - Loss: 0.0104
Batch 900/1236 - Loss: 0.1389
Batch 950/1236 - Loss: 0.0416
Batch 1000/1236 - Loss: 0.0041
Batch 1050/1236 - Loss: 0.0965
Batch 1100/1236 - Loss: 0.0349
Batch 1150/1236 - Loss: 0.0076
Batch 1200/1236 - Loss: 0.0933
Epoch 6 train loss: 0.0579
Validation Accuracy: 0.8731, F1-score: 0.8737


In [65]:
import os

save_dir = "C:/Models/rubert_finetuned"

os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Модель и токенизатор сохранены в: {save_dir}")



Модель и токенизатор сохранены в: C:/Models/rubert_finetuned


In [32]:
nltk.download('stopwords')
nltk.download('wordnet')

morph = pymorphy2.MorphAnalyzer()
lemm_en = WordNetLemmatizer()

stop_en = set(stopwords.words('english'))

stop_ru_custom = {
    "в","во","на","по","из","у","около","с","со","от","для","через","перед",
    "при","к","до","над","под","об","о","про","без","при","между",
    "и","а","но","да","же","ли","бы","то","ни","не","ну","что","как"
}

whitelist_short_words = {
    "ai", "vr", "os", "cpu", "gpu", "ram","ssd","hdd","pc","ios","php","js",
    "ip","dns","usa","ios","vr","xr","oc","ms","cs","dl","nlp","ml"
}


def clean_text(text):

    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    text = re.sub(r"@\w+", " ", text)

    text = emoji.replace_emoji(text, replace='')

    text = re.sub(r"[\[\]\(\)\{\}<>_\-=+\"“”'’•…#%&*^~$€@:;|/?]", " ", text)

    text = text.lower()

    words = text.split()

    clean_words = []
    for word in words:

        if word in whitelist_short_words:
            clean_words.append(word)
            continue

        if len(word) < 3:
            continue

        if word in stop_ru_custom:
            continue

        if word in stop_en:
            continue

        if re.match(r'^[а-я]+$', word):
            word = morph.parse(word)[0].normal_form

        elif re.match(r'^[a-z]+$', word):
            word = lemm_en.lemmatize(word)

        clean_words.append(word)

    return " ".join(clean_words)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kiru\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\kiru\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
import asyncio
from telethon import TelegramClient
import torch

async def fetch_channel_messages(channel_name, N=50):
    api_id = ID
    api_hash = HASH

    async with TelegramClient('session', api_id, api_hash) as client:
        posts = await client.get_messages(channel_name, limit=N)
        texts = [post.message for post in posts if post.message]
    return texts

def clean_and_tokenize(texts, tokenizer, MAX_LEN, device):
    texts_clean = [clean_text(t) for t in texts]
    if len(texts_clean) == 0:
        return None, None
    encodings = tokenizer(
        texts_clean,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    return encodings['input_ids'].to(device), encodings['attention_mask'].to(device), texts_clean

from collections import Counter

async def predict_channel_category(channel_name, model, tokenizer, le, device, N=50):
    texts = await fetch_channel_messages(channel_name, N)
    if len(texts) == 0:
        print("Нет доступных текстов в этом канале")
        return None

    input_ids, attention_mask, texts_clean = clean_and_tokenize(texts, tokenizer, MAX_LEN=128, device=device)
    if input_ids is None:
        return None

    model.eval()
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        categories = le.inverse_transform(preds.cpu().numpy())

    top_categories = [cat for cat, _ in Counter(categories).most_common(3)]
    print(f"Категория канала {channel_name}: {top_categories}")
    return top_categories

    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_test = AutoModelForSequenceClassification.from_pretrained("rubert_finetuned").to(device)
tokenizer_test = AutoTokenizer.from_pretrained("rubert_finetuned")
channel_cat = await predict_channel_category("@MAIuniversity", model_test, tokenizer_test, le, device, N=500)



The tokenizer you are loading from 'rubert_finetuned' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Категория канала @MAIuniversity: ['Образование', 'IT', 'Путешествия']
